# 07. Auditoría de Etiquetas y Detección de Solapamientos

Este cuaderno evalúa la calidad y consistencia del etiquetado histórico de Revista Anfibia. 
Vamos a cruzar los **tags libres** asignados por los redactores (columna `tagline`) con los **tópicos semánticos latentes** descubiertos por BERTopic.

**Objetivos:**
* Detectar solapamientos: Distintos tags que se usan para nombrar exactamente el mismo núcleo.
* Detectar ambigüedades: Tags genéricos que agrupan artículos sin ninguna relación temática real.
* Generar el diagnóstico cuantitativo necesario para proponer un sistema de re-etiquetado automático.

In [1]:
# 1. Parche de compatibilidad obligatorio (Numba/Coverage)
import coverage.types as ct

for name, fallback in {
    'Tracer': getattr(ct, 'TTracer', object),
    'TShouldTraceFn': object,
    'TShouldStartContextFn': object,
    'TFileDisposition': object,
}.items():
    if not hasattr(ct, name):
        setattr(ct, name, fallback)

# 2. Imports generales
import os
import re
import warnings
from pathlib import Path

import pandas as pd
from bertopic import BERTopic

warnings.filterwarnings("ignore", category=UserWarning)

# 3. Rutas
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "corpus_anfibia_lematizado.csv"
MODEL_PATH = PROJECT_ROOT / "outputs" / "bertopic_hibrido" / "bertopic_hibrido_model"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "analisis_etiquetas"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# 4. Carga de Datos
df = pd.read_csv(DATA_PATH)
df["document_text_original"] = df["document_text_original"].fillna("").astype(str).str.strip()
df["document_text_final"] = df["document_text_final"].fillna("").astype(str).str.strip()

# Filtrar vacíos y asignar doc_id
df = df[df["document_text_original"].ne("") & df["document_text_final"].ne("")].reset_index(drop=True)
if "doc_id" not in df.columns:
    df.insert(0, "doc_id", df.index)
else:
    df["doc_id"] = df.index

# 5. Carga de Modelo y Asignación de Tópicos
topic_model = BERTopic.load(MODEL_PATH)
topics, _ = topic_model.transform(df["document_text_final"].tolist())
df["topic"] = topics

topic_info = topic_model.get_topic_info().set_index("Topic")["Name"].to_dict()
df["topic_name"] = df["topic"].map(
    lambda t: ", ".join(topic_info.get(t, "Outlier").split("_")[1:4]) if t != -1 else "Outlier / Ruido"
)

print(f"Documentos procesados: {len(df):,}")

c:\Trabajos\anfibia-unsam\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Documentos procesados: 3,556


## 2. Desanidado de Tags
Los artículos suelen tener múltiples tags separados por comas. Desdoblamos el dataset para que cada fila represente la relación única entre un documento, un único tag y el tópico del modelo.

In [2]:
def split_tags(value):
    # Separar por comas o puntos y comas
    parts = re.split(r"[;,]", str(value))
    return [part.strip().lower() for part in parts if part.strip() and part.strip().lower() != "nan"]

rows = []
for _, r in df.iterrows():
    tags = split_tags(r["tagline"]) # Usamos tagline como la columna de etiquetas
    if not tags:
        tags = ["sin_etiqueta"]
    for tag in tags:
        rows.append({
            "doc_id": r["doc_id"],
            "title_text": r["title_text"],
            "tag": tag,
            "topic": r["topic"],
            "topic_name": r["topic_name"]
        })

df_tags = pd.DataFrame(rows)
print(f"Total de asignaciones tag-documento: {len(df_tags):,}")
display(df_tags.head())

Total de asignaciones tag-documento: 3,970


,doc_id,title_text,tag,topic,topic_name
0,0,La China invisible,feria la salada,-1,Outlier / Ruido
1,1,El rico que se cree Steve Jobs,gustavo grobocopatel,0,"político, política, discurso"
2,2,La vaca sagrada,sin_etiqueta,-1,Outlier / Ruido
3,3,Los arruinados del Paraná,adictos al juego,0,"político, política, discurso"
4,4,De Chile a Tacna: una historia de carniceros y...,sin_etiqueta,-1,Outlier / Ruido


## 3. Diagnóstico de Solapamientos 
Buscamos cuántos tags distintos se están usando para referirse al mismo núcleo semántico. 
Si el modelo agrupó notas bajo un mismo tópico muy claro pero hay decenas de tags diferentes.

In [3]:
# Agrupamos por tópico y contamos cuántos tags únicos y cuántas notas tiene cada uno
solapamiento_df = (
    df_tags[df_tags["topic"] != -1]
    .groupby(["topic", "topic_name"])
    .agg(
        total_notas=("doc_id", "nunique"),
        tags_unicos_usados=("tag", "nunique"),
        ejemplos_de_tags=("tag", lambda x: ", ".join(pd.Series.unique(x)[:5]))
    )
    .reset_index()
    .sort_values(by="tags_unicos_usados", ascending=False)
)

display(solapamiento_df.head(10))
solapamiento_df.to_csv(OUTPUT_DIR / "diagnostico_solapamiento_tags.csv", index=False)

,topic,topic_name,total_notas,tags_unicos_usados,ejemplos_de_tags
0,0,"político, política, discurso",1209,1215,"gustavo grobocopatel, adictos al juego, sin_et..."
1,1,"noticia, sociedad, tecnológico",88,84,"el experimento de zuckerberg, los trabajadores..."
2,2,"aborto, abortar, anticonceptivo",86,72,"género y crianza, fracaso de la ley de aborto,..."
3,3,"pandemia, epidemia, vacunar",63,70,"sin_etiqueta, medicamentos, esclerosis lateral..."
4,4,"papal, catolicismo, arzobispo",50,49,"espiritualidades, papa francisco, visiones del..."
7,7,"israel, netanyahu, israelí",24,25,"israel y palestina, conflicto en medio oriente..."
5,5,"arte, art, estética",22,22,"el mercado del arte, sin_etiqueta, comunidad a..."
6,6,"soviética, soviético, vladimir",20,18,"internacionales, rusia, la obra de svetlana al..."
9,9,"submarino, marino, océano",11,12,"sin_etiqueta, verano anfibio, malvinas, bolivi..."
8,8,"cientificismo, astrología, universo",8,9,"large hadron collider, carlo rovelli, el físic..."


## Tags por un topico puntual


In [10]:
# Inspección de solapamientos para un tópico puntual
TOPIC_ID = 1

df_detalle_topico = (
    df_tags[df_tags["topic"] == TOPIC_ID][["doc_id", "title_text", "tag"]]
    .drop_duplicates()
    .sort_values(by="title_text")
)

print(f"Listado de artículos y sus tags para el Tópico {TOPIC_ID} ({df[df['topic'] == TOPIC_ID]['topic_name'].iloc[0]}):")
display(df_detalle_topico.head(15))

# Exportar tabla completa de auditoría a CSV
df_tags.to_csv(OUTPUT_DIR / "auditoria_tags_por_documento.csv", index=False)
print("Archivo 'auditoria_tags_por_documento.csv' generado en outputs.")

Listado de artículos y sus tags para el Tópico 1 (noticia, sociedad, tecnológico):


,doc_id,title_text,tag
2773,2553,Alfabetismos aumentados para convivir con robots,futuro: claves para conseguir trabajo
3479,3150,Algoritmo mata pizarrón,el celular en la escuela
2473,2304,Antenas 5G y otras fantasías para evadir el covid,pandemia y conspiraciones
1354,1307,Apagá tu teléfono,cultura digital
2293,2151,Argentina y la fábrica de tecnología global,¿qué economía del conocimiento necesitamos?
3009,2756,Cambios en el mundo feliz de Silicon Valley,despidos masivos en la industria digital
994,972,Ciencia para qué,el recorte en el conicet
1475,1418,Cinco horas diarias mirando el teléfono,cultura digital
716,711,Clicks modernos,la agenda del público en la web
1346,1300,Conexión total,internet de las cosas


Archivo 'auditoria_tags_por_documento.csv' generado en outputs.


## 4. Visualización de Etiquetas (Dispersión)
Graficamos la relación entre la cantidad de notas de un tópico y la cantidad de tags únicos que los editores inventaron para esas notas. 
Una línea diagonal (relación 1 a 1) indica que no hay ninguna normalización: se inventa un tag nuevo casi por cada artículo publicado.

In [6]:
import plotly.express as px

# --- Gráfico 1: El panorama completo (Incluye el Tópico 0) ---
fig1 = px.scatter(
    solapamiento_df,
    x="total_notas",
    y="tags_unicos_usados",
    text="topic_name",
    size="total_notas",
    title="<b>1. Dispersión General (Incluye Tópico 0)</b>",
    labels={
        "total_notas": "Cantidad de Artículos en el Tópico",
        "tags_unicos_usados": "Cantidad de Tags Distintos Usados"
    },
    color="tags_unicos_usados",
    color_continuous_scale="Reds"
)
fig1.add_shape(type="line", x0=0, y0=0, x1=solapamiento_df["total_notas"].max(), y1=solapamiento_df["total_notas"].max(),
              line=dict(color="gray", dash="dash"), name="Relación 1 a 1")
fig1.update_traces(textposition='top left', marker=dict(line=dict(width=1, color='DarkSlateGrey')))
fig1.update_layout(height=500, width=1000, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='#ffffff')
fig1.update_xaxes(showgrid=True, gridcolor="#ebebeb")
fig1.update_yaxes(showgrid=True, gridcolor="#ebebeb")

# --- Gráfico 2: Zoom al resto de los tópicos (Filtramos el Tópico 0) ---
df_zoom = solapamiento_df[solapamiento_df["topic"] != 0].copy()

fig2 = px.scatter(
    df_zoom,
    x="total_notas",
    y="tags_unicos_usados",
    text="topic_name",
    size="total_notas",
    title="<b>2. Dispersión Detallada (Excluyendo el Tópico 0)</b>",
    labels={
        "total_notas": "Cantidad de Artículos en el Tópico",
        "tags_unicos_usados": "Cantidad de Tags Distintos Usados"
    },
    color="tags_unicos_usados",
    color_continuous_scale="Blues"
)
fig2.add_shape(type="line", x0=0, y0=0, x1=df_zoom["total_notas"].max(), y1=df_zoom["total_notas"].max(),
              line=dict(color="gray", dash="dash"), name="Relación 1 a 1")
fig2.update_traces(textposition='top center', marker=dict(line=dict(width=1, color='DarkSlateGrey')))
fig2.update_layout(height=550, width=1000, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='#ffffff')
fig2.update_xaxes(showgrid=True, gridcolor="#ebebeb")
fig2.update_yaxes(showgrid=True, gridcolor="#ebebeb")

# Guardar ambos
fig1.write_html(OUTPUT_DIR / "02_dispersion_tags_completo.html")
fig2.write_html(OUTPUT_DIR / "02_dispersion_tags_zoom.html")

# Mostrar
fig1.show()
fig2.show()

## 5. Auditoría (Hapax Legomena)
Analizamos cuántos tags fueron creados para ser usados **una sola vez** en toda la historia de la revista y nunca más se volvieron a asociar a otro texto.

In [11]:
# Conteo de frecuencia de uso por tag
tag_counts = df_tags[df_tags["tag"] != "sin_etiqueta"]["tag"].value_counts()

tags_unicos_total = len(tag_counts)
tags_una_vez = (tag_counts == 1).sum()
pct_una_vez = (tags_una_vez / tags_unicos_total) * 100

print(f"Total de tags únicos creados en Anfibia: {tags_unicos_total:,}")
print(f"Tags usados en exactamente 1 artículo: {tags_una_vez:,} ({pct_una_vez:.1f}%)")
print(f"Tags usados en 5 o más artículos: {(tag_counts >= 5).sum():,} ({((tag_counts >= 5).sum() / tags_unicos_total)*100:.1f}%)")

# Mostrar los tags más sobreutilizados vs una muestra de tags huérfanos
df_top_tags = tag_counts.head(10).reset_index()
df_top_tags.columns = ["Tag", "Cantidad de Artículos"]
print("\nTop 10 Tags más recurrentes:")
display(df_top_tags)

Total de tags únicos creados en Anfibia: 3,341
Tags usados en exactamente 1 artículo: 3,152 (94.3%)
Tags usados en 5 o más artículos: 35 (1.0%)

Top 10 Tags más recurrentes:


,Tag,Cantidad de Artículos
0,adelanto,32
1,cultura digital,18
2,o sea,16
3,aborto legal,13
4,comunidad anfibia,11
5,brasil 2014,10
6,homenaje a lemebel,10
7,concurso federal de relatos,9
8,masacre de orlando,9
9,talleres anfibia 2017,9


### Conclusiones:

Las etiquetas actuales no operan como un metodo de union/navegación, sino como un espacio libre para definir lo que es el articulo.

#### 1. Diagnóstico del Etiquetado Histórico
* **Hiperfragmentación extrema:** De **3.341 tags únicos** registrados, el **94.3% (3.152 tags)** fue utilizado en exactamente un único artículo.
* **Inexistencia de red temática:** Apenas el **1.0% de las etiquetas** se reutilizó en 5 o más notas, y los términos más frecuentes responden a formatos editoriales (*"adelanto"*, *"o sea"*) antes que a descriptores temáticos.
* **Pérdida de archivo:** En temas de alta densidad semántica (como el debate sobre el aborto, con 86 notas identificadas por el modelo), existen más de 70 etiquetas distintas, lo que fragmenta la cobertura e impide al lector recuperar el histórico mediante la interfaz web tradicional.

#### 2. Implicancias para el Desarrollo (Fase 2 y 3)
* **Desacople de metadatos:** Queda demostrado empíricamente que el futuro **sistema de recomendación** no puede apoyarse en los tags de WordPress. La recomendación de notas similares y la nube de exploración deben construirse estrictamente sobre la **similitud semántica (distancia coseno)** proyectada en el espacio latente de embeddings.
* **Propuesta de Re-etiquetado:** La solución técnica no consiste en sugerir palabras libres, sino en utilizar los tópicos de alta coherencia semántica como **etiquetas canónicas normalizadas** para ordenar el archivo y clasificar automáticamente los nuevos ingresos desde la API.

In [13]:
display(df_tags[df_tags["tag"] == "o sea"][["title_text", "topic_name"]])

,title_text,topic_name
3403,La pena del morfi,Outlier / Ruido
3415,El topo en su laberinto,"político, política, discurso"
3425,La casta está de rosca,"político, política, discurso"
3435,Estocolmo siempre estuvo lejos,"político, política, discurso"
3442,El niño mimado del Estado,Outlier / Ruido
3449,El extraño mundo de Javo el viajero,Outlier / Ruido
3457,Fútbol SAD: sin alma deportiva,"político, política, discurso"
3463,La fiesta de los espías,Outlier / Ruido
3470,Atrapado en libertad,Outlier / Ruido
3476,A naufragar,Outlier / Ruido
